# Stage C 03n — gated E100-minus-E25 warm-start continuation


In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='__PINNED_COMMIT__'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
DATASET_NAME='nonoverlap_6mer_v1'
TAXONOMY_MANIFEST=f'{DRIVE_ROOT}/stage_c_dataset/manifests/accession_manifest.parquet'
ACCESSION_MANIFEST=TAXONOMY_MANIFEST
ANI_MEMBERSHIP=f'{DRIVE_ROOT}/stage_c_dataset/manifests/ani99_membership.parquet'
ANI_PAIRS=f'{DRIVE_ROOT}/stage_c_dataset/manifests/skani_triangle.tsv'
RUN_NAME='c20_v3_medium_adaptive_e100_increment'
RUN_ID='medium_adaptive_e100_increment_v1'


In [ ]:
from pathlib import Path
from google.colab import drive
import json, shutil, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/DATASET_NAME
panels=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3/panels'
PROTOCOL=repo/'studies/stage_c_ecoli_medium_deep_memory_v3/protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
def run_logged(root,label,command):
    root.mkdir(parents=True,exist_ok=True)
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(root),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (root/'FAILED.txt',root/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-20000:])
        raise


In [ ]:
deep_flags=['--memory-architecture','paper_residual_mlp_v2','--memory-depth','2','--memory-expansion-factor','4','--memory-projection-convolution-kernel','4','--memory-normalize-queries-and-keys','--memory-gate-granularity','per_layer_channel','--memory-recurrence-policy','paper_exact','--memory-surprise-clip-norm','none','--memory-alpha-initial','0.001','--memory-eta-initial','0.9','--memory-theta-initial','0.001','--memory-associative-loss-reduction','sum','--memory-max-gradient-rms','none','--memory-max-gradient-rms-ratio','none','--memory-theta-max','1.0']
import torch
gate=Path(DRIVE_ROOT)/'runs/c19_v3_medium_adaptive_e25/scale_analysis_v1/gate/scale_gate.json'
if not json.loads(gate.read_text())['proceed']: raise RuntimeError('E25 gate says STOP; 03n is intentionally blocked.')
q=json.loads((Path(DRIVE_ROOT)/'runs/c18_v3_medium_a100_qualification/qualification_selection.json').read_text())
parent=Path(DRIVE_ROOT)/'runs/c19_v3_medium_adaptive_e25/latest.pt'; root=Path(DRIVE_ROOT)/'runs'/RUN_NAME
command=['seqtrainer-titans-stage-c-train','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'e100_additions.json'),'--validation-panel-manifest',str(panels/'validation.json'),'--run-dir',str(root),'--warm-start-checkpoint',str(parent),'--no-resume','--memory-mode','adaptive','--horizon','3','--batch-size',str(q['batch_size']),'--require-panel-completion','--scheduler-policy','stateful_rotation','--scheduler-burst-segments','96','--checkpoint-every','250','--learning-rate','3e-5','--min-learning-rate','3e-6','--lr-warmup-bases','2000000','--lr-decay-bases','100000000','--weight-decay','0.1','--gradient-clip-norm','0.5','--activation',q['activation'],'--block-count','12','--d-model','256','--num-heads','8','--persistent-tokens','4',*deep_flags,'--protocol',str(PROTOCOL),'--run-id',RUN_ID]
run_logged(root,'train_e100_increment',command)
run_logged(root,'resume_verify_e100',['seqtrainer-titans-stage-c-resume-verify','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'e100_additions.json'),'--checkpoint',str(root/'latest.pt'),'--output',str(root/'resume_verification.json'),'--device','cuda'])
print('SHARE THIS DIRECTORY:',root)
